# 02 — YOLO26 Training (Ultralytics ≥8.4.4)
**Drone Detection | ITMO Diploma Thesis**

Drop-in аналог `02_yolov12_train.ipynb`: те же `data.yaml` и API, другие чекпоинты:
- `yolo26n.pt` (nano)
- `yolo26s.pt` (small)

Веса сохраняются как `yolo26n_drone_best.pt` / `yolo26s_drone_best.pt`. Метрики **добавляются** в `results/yolo_metrics.json` (ключи `yolo26n`, `yolo26s`), не затирая `yolo12n`/`yolo12s`, если вы уже гоняли YOLOv12.

**Референс:** `reference/YOLO26_Tutorial.ipynb` (Ultralytics).

**I/O:** как в `02_yolov12_train.ipynb` — лейблы отдельной ячейкой (zip), **CELL 3** копирует только **`images`** с Drive.

In [ ]:
# ── CELL 1: Install & check GPU ────────────────────────────────────────────────
!uv pip install ultralytics
import ultralytics; ultralytics.checks()
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU detected! Switch runtime to GPU in Runtime → Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 3: только images с Drive (лейблы — ячейкой выше из zip) ───────────────
import os
import time
import json
import shutil
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
DRIVE_DATA = DRIVE_ROOT / 'prepared' / 'yolo'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
RESULTS_DIR = DRIVE_ROOT / 'results'

LOCAL_DATA = Path('/content/drone_yolo_local')
LOCAL_IMAGES = LOCAL_DATA / 'images'
DRIVE_IMAGES = DRIVE_DATA / 'images'
FORCE_RECOPY_IMAGES = False  # True: удалить локальный images/ и скопировать с Drive снова

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_IMAGES.is_dir(), f'На Drive нет папки: {DRIVE_IMAGES}'

LOCAL_DATA.mkdir(parents=True, exist_ok=True)

if LOCAL_IMAGES.exists() and FORCE_RECOPY_IMAGES:
    shutil.rmtree(LOCAL_IMAGES)

if not LOCAL_IMAGES.exists():
    print('Копирование images: Drive → /content/drone_yolo_local/images …')
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_IMAGES, LOCAL_IMAGES)
    print(f'Готово за {time.perf_counter() - t0:.1f} с')
else:
    print('images уже есть:', LOCAL_IMAGES, '(FORCE_RECOPY_IMAGES=True — перекопировать)')

DATA_DIR = LOCAL_DATA
YAML_PATH = DATA_DIR / 'data.yaml'

_root = DATA_DIR.resolve()
_imgs = _root / 'images'

def _split(name: str) -> str | None:
    p = _imgs / name
    return f'images/{name}' if p.is_dir() else None

_train = _split('train')
_val = _split('val')
_test = _split('test')

if _train is None:
    _listing = sorted(p.name for p in _imgs.iterdir()) if _imgs.is_dir() else []
    raise FileNotFoundError(
        f'Нет images/train в {_imgs}. Содержимое images/: {_listing}'
    )
if _val is None:
    _listing = sorted(p.name for p in _imgs.iterdir()) if _imgs.is_dir() else []
    raise FileNotFoundError(
        f'Нет images/val в {_imgs}. Содержимое images/: {_listing}'
    )
if _test is None:
    _test = _val
    print('Предупреждение: images/test нет — test в data.yaml = val (для train ок; для финального теста добавьте split).')

_yaml_fixed = f'''# Local Colab — fast I/O (dataset copied from Drive in this notebook)
path: {_root.as_posix()}
train: {_train}
val: {_val}
test: {_test}
nc: 4
names: ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
'''
YAML_PATH.write_text(_yaml_fixed, encoding='utf-8')
for _rel in (_train, _val, _test):
    _p = _root / _rel
    assert _p.is_dir(), f'После записи data.yaml нет папки: {_p}'
print(f'data.yaml → path = {_root.as_posix()} ✓ (train={_train}, val={_val}, test={_test})')

_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
print(f'GPU VRAM ~{_vram_gb:.1f} GB — при OOM уменьшите batch в TRAIN_CONFIG')

## Конфигурация обучения
Те же гиперпараметры, что и для YOLOv12 (`02_yolov12_train.ipynb`): локальный датасет, `workers=8`, `cache='disk'`.

In [ ]:
# ── CELL 4: Training config ───────────────────────────────────────────────────
TRAIN_CONFIG = {
    'data':       str(YAML_PATH),
    'epochs':     50,
    'imgsz':      640,
    'batch':      16,          # T4: 16; на 16+ GB VRAM после копии на /content можно 24–32
    'device':     0,
    'workers':    8,
    'cache':      'disk',      # при ошибке API уберите ключ или поставьте False
    'patience':   10,
    'save':       True,
    'pretrained': True,
    'optimizer':  'AdamW',
    'lr0':        0.001,
    'lrf':        0.01,
    'momentum':   0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'hsv_h':      0.015,
    'hsv_s':      0.7,
    'hsv_v':      0.4,
    'degrees':    10.0,
    'translate':  0.1,
    'scale':      0.5,
    'fliplr':     0.5,
    'mosaic':     1.0,
    'mixup':      0.1,
    'copy_paste': 0.1,
}
print('Training config ready.')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

## 1. Обучение YOLO26n (nano)

In [ ]:
# ── CELL 5: Train YOLO26n ─────────────────────────────────────────────────────
print('=' * 60)
print('Training YOLO26n (nano)')
print('=' * 60)

model_n = YOLO('yolo26n.pt')

results_n = model_n.train(
    name='yolo26n_drone',
    project='/content/runs',
    **TRAIN_CONFIG
)

best_n_src = Path(model_n.trainer.save_dir) / 'weights' / 'best.pt'
best_n_dst = WEIGHTS_DIR / 'yolo26n_drone_best.pt'
assert best_n_src.is_file(), f'Нет {best_n_src}'
shutil.copy(best_n_src, best_n_dst)
print(f'\nBest weights saved → {best_n_dst} (from {best_n_src.parent.parent.name})')

## 2. Обучение YOLO26s (small)

In [ ]:
# ── CELL 6: Train YOLO26s ───────────────────────────────────────────────────────
print('=' * 60)
print('Training YOLO26s (small)')
print('=' * 60)

model_s = YOLO('yolo26s.pt')

results_s = model_s.train(
    name='yolo26s_drone',
    project='/content/runs',
    **TRAIN_CONFIG,
    save_period=1,  # чекпоинт каждую эпоху → .../weights/epochN.pt (+ best.pt / last.pt)
)

best_s_src = Path(model_s.trainer.save_dir) / 'weights' / 'best.pt'
best_s_dst = WEIGHTS_DIR / 'yolo26s_drone_best.pt'
assert best_s_src.is_file(), f'Нет {best_s_src}'
shutil.copy(best_s_src, best_s_dst)
print(f'\nBest weights saved → {best_s_dst} (from {best_s_src.parent.parent.name})')

## 3. Валидация и метрики

In [ ]:
# ── CELL 7: Validate both YOLO26 models on test set ─────────────────────────────
def validate_yolo(weights_path: Path, data_yaml: Path,
                  split: str = 'test') -> dict:
    """Run YOLO validation and return metrics dict."""
    model = YOLO(str(weights_path))
    metrics = model.val(data=str(data_yaml), split=split, device=0, verbose=True)
    return {
        'mAP50':      round(float(metrics.box.map50),    4),
        'mAP50_95':   round(float(metrics.box.map),      4),
        'precision':  round(float(metrics.box.mp),       4),
        'recall':     round(float(metrics.box.mr),       4),
        'per_class_mAP50': metrics.box.ap50.tolist(),
    }


metrics_n = validate_yolo(WEIGHTS_DIR / 'yolo26n_drone_best.pt', YAML_PATH)
metrics_s = validate_yolo(WEIGHTS_DIR / 'yolo26s_drone_best.pt', YAML_PATH)

print('\n─── YOLO26n ───')
for k, v in metrics_n.items():
    print(f'  {k}: {v}')
print('\n─── YOLO26s ───')
for k, v in metrics_s.items():
    print(f'  {k}: {v}')

In [ ]:
# ── CELL 8: Measure FPS ─────────────────────────────────────────────────────────
def measure_fps(weights_path: Path, test_img_dir: Path,
                n_runs: int = 100) -> float:
    """Measure average inference FPS on GPU."""
    model = YOLO(str(weights_path))
    imgs = (list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))[:n_runs]
    if not imgs:
        return 0.0
    for _ in range(5):
        model.predict(str(imgs[0]), device=0, verbose=False)
    start = time.perf_counter()
    for img in imgs:
        model.predict(str(img), device=0, verbose=False)
    elapsed = time.perf_counter() - start
    return round(len(imgs) / elapsed, 1)


test_img_dir = DATA_DIR / 'images' / 'test'
fps_n = measure_fps(WEIGHTS_DIR / 'yolo26n_drone_best.pt', test_img_dir)
fps_s = measure_fps(WEIGHTS_DIR / 'yolo26s_drone_best.pt', test_img_dir)
print(f'YOLO26n FPS: {fps_n}')
print(f'YOLO26s FPS: {fps_s}')

In [ ]:
# ── CELL 9: Merge YOLO26 metrics into yolo_metrics.json ─────────────────────────
CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']

out = RESULTS_DIR / 'yolo_metrics.json'
existing = {}
if out.exists():
    with open(out) as f:
        existing = json.load(f)

existing['yolo26n'] = {
    **metrics_n, 'fps': fps_n,
    'size_mb': round((WEIGHTS_DIR / 'yolo26n_drone_best.pt').stat().st_size / 1e6, 1),
}
existing['yolo26s'] = {
    **metrics_s, 'fps': fps_s,
    'size_mb': round((WEIGHTS_DIR / 'yolo26s_drone_best.pt').stat().st_size / 1e6, 1),
}
existing['class_names'] = CLASS_NAMES

with open(out, 'w') as f:
    json.dump(existing, f, indent=2)
print(f'Metrics merged → {out}')
print('(Keys yolo12n/yolo12s preserved if they were already present.)')

In [ ]:
# ── CELL 10: Plot training curves ───────────────────────────────────────────────
def plot_training_curves(run_dir: str, model_name: str) -> None:
    """Plot loss and mAP curves from YOLO results.csv."""
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        print(f'results.csv not found: {csv_path}')
        return
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'{model_name} Training Curves', fontsize=14, fontweight='bold')

    plots = [
        ('train/box_loss',  'Train Box Loss',     axes[0, 0]),
        ('train/cls_loss',  'Train Class Loss',   axes[0, 1]),
        ('train/dfl_loss',  'Train DFL Loss',     axes[0, 2]),
        ('metrics/mAP50(B)', 'mAP@0.5',            axes[1, 0]),
        ('metrics/mAP50-95(B)', 'mAP@0.5:0.95',   axes[1, 1]),
        ('metrics/precision(B)', 'Precision',    axes[1, 2]),
    ]

    for col, title, ax in plots:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color='#9b59b6', linewidth=2)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.set_title(f'{title} (N/A)')

    plt.tight_layout()
    save_path = RESULTS_DIR / f'{model_name}_curves.png'
    plt.savefig(save_path, dpi=150)
    print(f'Curves saved → {save_path}')
    plt.show()


def _latest_yolo_run(prefix: str) -> str:
    root = Path('/content/runs')
    matches = sorted(root.glob(f'{prefix}*'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f'Нет каталога {root}/{prefix}*')
    return str(matches[0])


plot_training_curves(_latest_yolo_run('yolo26n_drone'), 'YOLO26n')
plot_training_curves(_latest_yolo_run('yolo26s_drone'), 'YOLO26s')

In [ ]:
# ── CELL 11: Export to ONNX ─────────────────────────────────────────────────────
# Ultralytics пишет .onnx рядом с .pt (то же базовое имя) — shutil.copy не нужен.
for model_name, weights in [('yolo26n', WEIGHTS_DIR / 'yolo26n_drone_best.pt'),
                            ('yolo26s', WEIGHTS_DIR / 'yolo26s_drone_best.pt')]:
    model = YOLO(str(weights))
    model.export(format='onnx', imgsz=640, simplify=True)
    onnx_path = weights.with_suffix('.onnx')
    if onnx_path.exists():
        print(f'{model_name} ONNX → {onnx_path}')
    else:
        print(f'Warning: ONNX not found after export: {onnx_path}')

In [ ]:
# ── CELL 12: Quick prediction visualization ─────────────────────────────────────
model_s_loaded = YOLO(str(WEIGHTS_DIR / 'yolo26s_drone_best.pt'))
test_imgs = list((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
fig.suptitle('YOLO26s Predictions on Test Set', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes, test_imgs):
    results = model_s_loaded.predict(str(img_path), device=0, conf=0.25, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[..., ::-1])
    ax.axis('off')
    ax.set_title(img_path.name[:25], fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'yolo26s_predictions.png', dpi=120)
plt.show()
print('Done! Next → 04_evaluation.ipynb (or 03_faster_rcnn_train.ipynb if not done yet)')